# MDR-TS v20.7
**Temporal-Station Model — Leave-One-Station-Out (All Stations)**

**Author:** Jakob Balkovec  
**Affiliation:** Seattle University, Computer Science  
**Project:** MDR  
**Notebook Type:** Training & Evaluation  

---

## What's new vs v20.6
- **Full LOSO sweep** — all 5 stations are held out in turn: Darrington, Quinault, SourdoughGulch_WA_985, Spokane, Touchet_WA_824
- For each holdout the model trains on the remaining 4 stations (same 3-regime XGBoost architecture as v20.6)
- Results are aggregated into a single cross-station comparison table

## Spatial generalization goal
How well does the 3-regime XGBoost generalise to **each** unseen station individually?

## 0. Imports

In [ ]:
import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from xgboost import XGBRegressor, XGBClassifier

import joblib
import statsmodels.api as sm

project_root = os.path.abspath("../../")
if project_root not in sys.path:
    sys.path.append(project_root)

from Utils.dashboard import metrics_dashboard

import warnings
warnings.filterwarnings("ignore")

print("imports loaded")

## 1. Config

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

VERSION    = "v20"
SUBVERSION = "v20.7"

PROJECT_ROOT = "/Users/kerrycheon/PycharmProjects/MDR-Project"
DATA_ROOT    = f"{PROJECT_ROOT}/Temporal/Pipeline/data"
SPLIT_ROOT   = f"{DATA_ROOT}/splits"
OUTPUT_ROOT  = f"{PROJECT_ROOT}/Models/Temporal/{VERSION}/{SUBVERSION}"
V205_ROOT    = f"{PROJECT_ROOT}/Models/Temporal/{VERSION}/v20.5"  # for loading classifier

SPLIT = "derived_8.0"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

ALL_STATIONS = ["Darrington", "Quinault", "SourdoughGulch_WA_985", "Spokane", "Touchet_WA_824"]

print(f"Random seed:    {SEED}")
print(f"Output:         {OUTPUT_ROOT}")
print(f"LOSO stations:  {ALL_STATIONS}")

## 2. Feature columns

In [ ]:
TARGET_COL     = "soil_moisture_5cm"
KEEP_META_COLS = ["station_id", "date", "longitude", "latitude"]

FEATURE_COLS_BASE = [
    "SMAP_sm_pm_interp_ema02",
    "SMAP_sm_interp_grad7",
    "SMAP_ampm_diff_interp",
    "G_API",
    "G_rain_sum_3d",
    "G_rain_sum_7d",
    "V_ema_G_API_kobs7",
    "V_ema_G_API_kobs14",
    "V_ema_G_API_kobs30",
    "V_rollmean_G_API_kobs7",
    "V_rollmean_G_API_kobs14",
    "A_d_E_SAR_diff_kobs14",
    "V_ema_LST_modis_kobs7",
    "A_d_LST_modis_kobs14",
    "V_rollmin_LST_modis_kobs30",
    "V_rollmean_s2_b11_kobs7",
    "year_frac", "sin_year", "cos_year",
    "API_x_year", "SMAP_x_year",
    "slope", "elev",
    "K_slope_sin", "K_slope_cos", "K_aspect_cos",
    "J_clay_wfrac_b0", "J_sand_wfrac_b0",
]

FEATURE_COLS_DRY = [
    "SMAP_sm_pm_interp_ema02",
    "SMAP_sm_interp_grad7",
    "SMAP_sm_interp_diff1",
    "A_d_SMAP_sm_interp_kobs14",
    "V_ema_LST_modis_kobs7",
    "V_rollmin_LST_modis_kobs30",
    "A_d_LST_modis_kobs14",
    "slope", "elev",
    "K_slope_sin", "K_slope_cos", "K_aspect_cos",
    "J_clay_wfrac_b0", "J_sand_wfrac_b0",
    "G_API",
    "V_ema_G_API_kobs14",
    "C_lag_G_API_kobs1",
    "V_rollmean_s2_b11_kobs7",
    "year_frac", "sin_year", "cos_year",
]

FEATURE_COLS_WET = [
    "SMAP_sm_interp_diff1",
    "SMAP_sm_interp_rollstd7",
    "SMAP_sm_interp_rollrange7",
    "SMAP_sm_interp_pctchg",
    "A_d_SMAP_sm_interp_kobs7",
    "A_grad_SMAP_sm_interp_kobs7",
    "A_pct_SMAP_sm_interp",
    "G_API",
    "G_rain_sum_3d",
    "G_rain_sum_7d",
    "V_rollstd_G_API_kobs7",
    "V_rollcv_G_API_kobs7",
    "A_d_G_API_kobs7",
    "A_d_E_SAR_diff_kobs1",
    "A_d_E_SAR_diff_kobs7",
    "A_grad_E_SAR_diff_kobs7",
    "A_grad_E_SAR_ratio_kobs7",
    "V_rollstd_E_SAR_diff_kobs7",
    "V_rollstd_E_SAR_ratio_kobs7",
    "V_rollstd_F_NDMI_kobs7",
    "A_d_F_NDMI_kobs7",
    "year_frac", "sin_year", "cos_year",
    "slope", "elev",
]

print("Feature cols locked — BASE:", len(FEATURE_COLS_BASE),
      "DRY:", len(FEATURE_COLS_DRY), "WET:", len(FEATURE_COLS_WET))

## 3. Model hyperparameters

In [ ]:
XGB_PARAMS_DRY = dict(
    objective="reg:absoluteerror", random_state=SEED, n_jobs=-1,
    subsample=0.9, colsample_bytree=0.8, max_depth=8, min_child_weight=2,
    n_estimators=5500, learning_rate=0.04, reg_lambda=1.5, reg_alpha=0.03, gamma=0.0,
)

XGB_PARAMS_TRANSITION = dict(
    objective="reg:absoluteerror", random_state=SEED, n_jobs=-1,
    max_depth=7, min_child_weight=5, subsample=0.9, colsample_bytree=0.85,
    n_estimators=8000, learning_rate=0.03, reg_lambda=3.0, reg_alpha=0.05,
)

XGB_PARAMS_WET = dict(
    objective="reg:squarederror", random_state=SEED, n_jobs=-1,
    max_depth=10, min_child_weight=1, subsample=1.0, colsample_bytree=0.9,
    n_estimators=6000, learning_rate=0.03, reg_lambda=0.3, reg_alpha=0.0,
)

T1, T2 = 0.20, 0.313  # regime thresholds
STABLE_THRESH = 0.60

print("Hyperparameters locked")

## 4. Helper functions

In [ ]:
def label_regime(y, t1=T1, t2=T2):
    y = np.asarray(y).ravel()
    out = np.zeros(len(y), dtype=int)
    out[(y > t1) & (y <= t2)] = 1
    out[y > t2] = 2
    return out


def get_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    err    = y_true - y_pred
    return {
        "n":      int(len(y_true)),
        "r2":     float(r2_score(y_true, y_pred)),
        "mae":    float(mean_absolute_error(y_true, y_pred)),
        "rmse":   float(root_mean_squared_error(y_true, y_pred)),
        "ubrmse": float(np.std(err)),
        "bias":   float(np.mean(err)),
    }


def make_final_pred(base_pred, pred_trans, pred_wet, p_stable, t1=T1, t2=T2, thresh=STABLE_THRESH):
    mask_dry   = base_pred <= t1
    mask_trans = (base_pred > t1) & (base_pred <= t2)
    mask_wet   = base_pred > t2
    is_stable  = p_stable >= thresh

    pred = np.zeros_like(base_pred, dtype=float)
    pred[mask_dry]                    = base_pred[mask_dry]
    pred[mask_wet]                    = pred_wet[mask_wet]
    pred[mask_trans &  is_stable]     = base_pred[mask_trans & is_stable]
    pred[mask_trans & ~is_stable]     = pred_trans[mask_trans & ~is_stable]
    return pred


def clf_proba(df, clf, imputer, cols, clip_q=0.001):
    X = df[cols].copy().replace([np.inf, -np.inf], np.nan)
    if clip_q:
        lo = X.quantile(clip_q,   axis=0, numeric_only=True)
        hi = X.quantile(1-clip_q, axis=0, numeric_only=True)
        X  = X.clip(lower=lo, upper=hi, axis=1)
    Xi = imputer.transform(X)
    return clf.predict_proba(Xi)[:, 1]


print("Helpers defined")

## 5. Load raw splits + regime classifier

In [ ]:
train_raw = pd.read_csv(f"{SPLIT_ROOT}/{SPLIT}/train.csv")
val_raw   = pd.read_csv(f"{SPLIT_ROOT}/{SPLIT}/val.csv")
test_raw  = pd.read_csv(f"{SPLIT_ROOT}/{SPLIT}/test.csv")

print("Raw split sizes:")
print(f"  train: {len(train_raw):,}  val: {len(val_raw):,}  test: {len(test_raw):,}")
print("Stations:", sorted(train_raw['station_id'].unique()))

In [ ]:
MODEL_PATH   = f"{V205_ROOT}/regime_classifier_xgb.json"
IMPUTER_PATH = f"{V205_ROOT}/stable_classifier_imputer.pkl"
COLS_PATH    = f"{V205_ROOT}/stable_classifier_cols.pkl"

USE_CLF = all(os.path.exists(p) for p in [MODEL_PATH, IMPUTER_PATH, COLS_PATH])

if USE_CLF:
    clf      = XGBClassifier()
    clf.load_model(MODEL_PATH)
    imputer  = joblib.load(IMPUTER_PATH)
    CLF_COLS = joblib.load(COLS_PATH)
    print(f"Regime classifier loaded | CLF_COLS: {len(CLF_COLS)}")
else:
    clf = imputer = CLF_COLS = None
    print("WARNING: classifier files not found — p_stable will be 0 for all rows (oracle fallback)")

## 6. LOSO loop

In [ ]:
loso_results = []   # list of dicts — one per holdout station
loso_preds   = {}   # station_id -> (y_true, y_pred, dates) for plotting

for HOLDOUT in ALL_STATIONS:
    print(f"\n{'='*60}")
    print(f"  HOLDOUT: {HOLDOUT}")
    print(f"{'='*60}")

    # ── carve out holdout ──────────────────────────────────────────────────────
    holdout_df = pd.concat([
        train_raw[train_raw["station_id"] == HOLDOUT],
        val_raw  [val_raw  ["station_id"] == HOLDOUT],
        test_raw [test_raw ["station_id"] == HOLDOUT],
    ], ignore_index=True)

    train_df = train_raw[train_raw["station_id"] != HOLDOUT].reset_index(drop=True)
    val_df   = val_raw  [val_raw  ["station_id"] != HOLDOUT].reset_index(drop=True)

    trainval_df = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

    print(f"  trainval rows: {len(trainval_df):,}  |  holdout rows: {len(holdout_df):,}")
    print(f"  trainval stations: {sorted(trainval_df['station_id'].unique())}")

    # ── matrices ───────────────────────────────────────────────────────────────
    X_tv_base = trainval_df[FEATURE_COLS_BASE]
    X_tv_wet  = trainval_df[FEATURE_COLS_WET]
    y_tv      = trainval_df[TARGET_COL].values

    X_h_base = holdout_df[FEATURE_COLS_BASE]
    X_h_wet  = holdout_df[FEATURE_COLS_WET]
    y_h      = holdout_df[TARGET_COL].values

    reg_tv = label_regime(y_tv)
    mask_trans_tv = reg_tv == 1
    mask_wet_tv   = reg_tv == 2

    # ── dry / anchor ───────────────────────────────────────────────────────────
    print("  [1/3] training dry/anchor ...")
    xgb_dry = XGBRegressor(**XGB_PARAMS_DRY)
    xgb_dry.fit(X_tv_base, y_tv, verbose=0)
    pred_base_tv = xgb_dry.predict(X_tv_base)
    pred_base_h  = xgb_dry.predict(X_h_base)

    # ── augmented wet matrices ─────────────────────────────────────────────────
    X_tv_aug = np.column_stack([X_tv_wet, pred_base_tv])
    X_h_aug  = np.column_stack([X_h_wet,  pred_base_h])

    # ── transition specialist ──────────────────────────────────────────────────
    print("  [2/3] training transition specialist ...")
    xgb_trans = XGBRegressor(**XGB_PARAMS_TRANSITION)
    xgb_trans.fit(X_tv_aug[mask_trans_tv], y_tv[mask_trans_tv], verbose=0)
    pred_trans_h = xgb_trans.predict(X_h_aug)

    # ── wet specialist ─────────────────────────────────────────────────────────
    print("  [3/3] training wet specialist ...")
    xgb_wet = XGBRegressor(**XGB_PARAMS_WET)
    xgb_wet.fit(X_tv_aug[mask_wet_tv], y_tv[mask_wet_tv], verbose=0)
    pred_wet_h = xgb_wet.predict(X_h_aug)

    # ── regime classifier routing ──────────────────────────────────────────────
    if USE_CLF:
        p_stable_h = clf_proba(holdout_df, clf, imputer, CLF_COLS)
    else:
        p_stable_h = np.zeros(len(holdout_df), dtype=float)

    # ── final predictions ──────────────────────────────────────────────────────
    pred_final = make_final_pred(
        np.asarray(pred_base_h).ravel(),
        np.asarray(pred_trans_h).ravel(),
        np.asarray(pred_wet_h).ravel(),
        p_stable_h,
    )

    # ── overall metrics ────────────────────────────────────────────────────────
    m = get_metrics(y_h, pred_final)
    print(f"  R²={m['r2']:+.4f}  RMSE={m['rmse']:.4f}  ubRMSE={m['ubrmse']:.4f}  Bias={m['bias']:+.4f}")

    # ── per-regime metrics ─────────────────────────────────────────────────────
    reg_h = label_regime(y_h)
    per_regime = {}
    for rname, ridx in [("dry", 0), ("transition", 1), ("wet", 2)]:
        mask = reg_h == ridx
        if mask.sum() > 0:
            per_regime[rname] = get_metrics(y_h[mask], pred_final[mask])
        else:
            per_regime[rname] = None

    row = {"station": HOLDOUT, **{f"{k}": v for k, v in m.items()}}
    for rname, rm in per_regime.items():
        if rm:
            for k, v in rm.items():
                row[f"{rname}_{k}"] = v
    loso_results.append(row)

    # ── store preds for plots ──────────────────────────────────────────────────
    loso_preds[HOLDOUT] = (
        y_h,
        pred_final,
        pd.to_datetime(holdout_df["date"].values),
    )

print("\nLOSO loop complete")

## 7. Summary table

In [ ]:
from IPython.display import display

summary_df = pd.DataFrame(loso_results)

# ── Overall metrics table ──────────────────────────────────────────────────────
overall_cols = ["station", "n", "r2", "mae", "rmse", "ubrmse", "bias"]
display(
    summary_df[overall_cols].style
    .format({"n": "{:,}", "r2": "{:.4f}", "mae": "{:.4f}",
             "rmse": "{:.4f}", "ubrmse": "{:.4f}", "bias": "{:+.4f}"})
    .background_gradient(subset=["r2"], cmap="Greens")
    .background_gradient(subset=["rmse", "ubrmse"], cmap="Reds_r")
    .set_caption("v20.7 — LOSO Overall Metrics (all holdout stations)")
)

summary_df.to_csv(f"{OUTPUT_ROOT}/loso_all_stations_summary.csv", index=False)
print(f"Saved: loso_all_stations_summary.csv")

In [ ]:
# ── Per-regime R² table ────────────────────────────────────────────────────────
regime_rows = []
for r in loso_results:
    regime_rows.append({
        "station":       r["station"],
        "overall_r2":    r["r2"],
        "dry_r2":        r.get("dry_r2"),
        "transition_r2": r.get("transition_r2"),
        "wet_r2":        r.get("wet_r2"),
        "dry_n":         r.get("dry_n"),
        "transition_n":  r.get("transition_n"),
        "wet_n":         r.get("wet_n"),
    })

regime_df = pd.DataFrame(regime_rows)
fmt = {c: "{:.4f}" for c in ["overall_r2", "dry_r2", "transition_r2", "wet_r2"]}
fmt.update({c: "{:,}" for c in ["dry_n", "transition_n", "wet_n"]})

display(
    regime_df.style.format(fmt, na_rep="—")
    .background_gradient(subset=["overall_r2", "dry_r2", "transition_r2", "wet_r2"],
                         cmap="RdYlGn", vmin=-0.5, vmax=1.0)
    .set_caption("v20.7 — LOSO Per-Regime R² (all holdout stations)")
)

## 8. Visualisations

In [ ]:
# ── Scatter grid: predicted vs true ───────────────────────────────────────────
n_stations = len(ALL_STATIONS)
fig, axes = plt.subplots(1, n_stations, figsize=(4 * n_stations, 4), sharey=False)

colors = ["steelblue", "darkorange", "seagreen", "mediumpurple", "crimson"]

for ax, station, color in zip(axes, ALL_STATIONS, colors):
    y_true, y_pred, _ = loso_preds[station]
    r2 = r2_score(y_true, y_pred)
    ax.scatter(y_true, y_pred, s=6, alpha=0.35, color=color)
    lo = min(y_true.min(), y_pred.min()) - 0.01
    hi = max(y_true.max(), y_pred.max()) + 0.01
    ax.plot([lo, hi], [lo, hi], "k--", lw=0.8)
    ax.set_title(f"{station}\nR²={r2:.3f}", fontsize=9)
    ax.set_xlabel("True SM", fontsize=8)
    ax.set_ylabel("Pred SM", fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("v20.7 LOSO — Predicted vs True (each station held out)", fontsize=11)
plt.tight_layout()
plt.savefig(f"{OUTPUT_ROOT}/loso_scatter_all.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Residuals grid ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, n_stations, figsize=(4 * n_stations, 4), sharey=True)

for ax, station, color in zip(axes, ALL_STATIONS, colors):
    y_true, y_pred, _ = loso_preds[station]
    res = y_true - y_pred
    ax.scatter(y_true, res, s=6, alpha=0.35, color=color)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_title(f"{station}\nbias={np.mean(res):+.4f}", fontsize=9)
    ax.set_xlabel("True SM", fontsize=8)
    ax.set_ylabel("Residual", fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("v20.7 LOSO — Residuals (each station held out)", fontsize=11)
plt.tight_layout()
plt.savefig(f"{OUTPUT_ROOT}/loso_residuals_all.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Time-series grid ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(n_stations, 1, figsize=(16, 3 * n_stations), sharex=False)

for ax, station, color in zip(axes, ALL_STATIONS, colors):
    y_true, y_pred, dates = loso_preds[station]
    order = np.argsort(dates)
    ax.plot(dates[order], y_true[order], label="True",  color="steelblue", lw=1.1, alpha=0.8)
    ax.plot(dates[order], y_pred[order], label="Pred",  color=color,       lw=0.9, alpha=0.9)
    r2 = r2_score(y_true, y_pred)
    ax.set_title(f"{station}  (R²={r2:.3f})", fontsize=9)
    ax.set_ylabel("SM 5cm", fontsize=8)
    ax.legend(fontsize=7, loc="upper right")
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Date")
plt.suptitle("v20.7 LOSO — Time Series (each station held out)", fontsize=12)
plt.tight_layout()
plt.savefig(f"{OUTPUT_ROOT}/loso_timeseries_all.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Bar chart: R² per station ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

r2_vals   = [r["r2"]      for r in loso_results]
rmse_vals = [r["rmse"]    for r in loso_results]
stations  = [r["station"] for r in loso_results]

x = np.arange(len(stations))
bars = ax.bar(x, r2_vals, color=colors, alpha=0.85, edgecolor="k", linewidth=0.6)

for bar, r2, rmse in zip(bars, r2_vals, rmse_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"R²={r2:.3f}\nRMSE={rmse:.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(stations, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("R²")
ax.set_ylim(min(0, min(r2_vals)) - 0.05, 1.05)
ax.axhline(0, color="k", lw=0.7, linestyle="--")
ax.set_title("v20.7 LOSO — R² by Holdout Station", fontsize=11)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_ROOT}/loso_r2_bar.png", dpi=150, bbox_inches="tight")
plt.show()

---
_Jakob Balkovec_